In [40]:
from ipyleaflet import (
    Map,
    basemaps,
    TileLayer,
    GeoJSON,
    LayersControl,
    ScaleControl,
    FullScreenControl,
    WidgetControl,
    basemap_to_tiles
)

from ipywidgets import HTML
from IPython.display import display

import json
import requests
import polars as pl

import os
os.chdir("../app")

from shared import (
    get_tif_path,
    load_species_metadata,
    load_abundance_data,
    get_subregion_boundaries
)

In [41]:
birds = load_species_metadata()
abundances = load_abundance_data()
subregions = get_subregion_boundaries()

PRODUCTION_TILER_BASE = (
    "https://019e4735-507f-07a0-1ae5-b96da68b058b.share.connect.posit.cloud"
)

REGION_CENTERS = {
    "Alaska": [64, -149],
    "Canada": [55, -106],
    "Lower48": [47.0, -97.0]
}

# test tif
SPECIES_NAME = "American Robin"
REGION = "Canada"
YEAR = 2020

In [42]:
def url_exists(url: str) -> bool:
    try:
        r = requests.head(url, timeout=10)
        return r.status_code == 200
    except Exception:
        return False

def raster_statistics(url: str):

    if not url or not url_exists(url):
        return {"min": 0.0, "max": 1.0}

    try:

        encoded_cog = requests.utils.quote(url, safe="")

        stats_url = (
            f"{PRODUCTION_TILER_BASE}"
            f"/cog/statistics?url={encoded_cog}"
        )

        res = requests.get(stats_url, timeout=10).json()

        band_key = list(res.keys())[0]

        band_stats = res[band_key]

        return {
            "min": float(band_stats.get("min", 0.0)),
            "max": float(band_stats.get("max", 1.0))
        }

    except Exception as e:

        print("Statistics error:", e)

        return {
            "min": 0.0,
            "max": 1.0
        }

In [43]:
bird = birds.filter(
    pl.col("english") == SPECIES_NAME
)

species_id = bird.item(0, "id")

url = get_tif_path(
    species_id,
    REGION,
    YEAR
)

print(url)

if not url_exists(url):
    raise RuntimeError("Raster does not exist")

stats = raster_statistics(url)

rmin = stats["min"]
rmax = stats["max"]

print("Min:", rmin)
print("Max:", rmax)

http://206.12.92.143/data/dashboard/AMRO/Canada/AMRO_Canada_2020.tif
Min: 0.0
Max: 1.0322922468185425


In [ ]:
map_center = REGION_CENTERS.get(
    REGION,
    [60.0, -110.0]
)

# basemaps
positron = basemap_to_tiles(
    basemaps.CartoDB.Positron
)
positron.base = True
positron.name = "Positron"

osm = basemap_to_tiles(
    basemaps.OpenStreetMap.Mapnik
)
osm.base = True
osm.name = "OSM"

esri = basemap_to_tiles(
    basemap=basemaps.Esri.WorldImagery
)
esri.base = True
esri.name = "Imagery"

# create map
m = Map(
    layers=[esri, positron, osm],
    center=map_center,
    zoom=4
)

encoded_cog = requests.utils.quote(
    url,
    safe=""
)

tile_string = (
    f"{PRODUCTION_TILER_BASE}"
    f"/cog/tiles/{{z}}/{{x}}/{{y}}.png"
    f"?url={encoded_cog}"
    f"&colormap_name=ylgn"
    f"&rescale={rmin},{rmax}"
)

mean_density = TileLayer(
    url=tile_string,
    name="Mean Density",
    opacity=0.75
)

def filter_subregions(gdf, region: str):

    if region == "Canada":
        return gdf[gdf["bcr"].str.startswith("can")]

    if region == "Lower48":
        return gdf[gdf["bcr"].str.startswith("usa")]

    if region == "Alaska":
        return gdf[gdf["bcr"] == "Alaska"]
    
    return gdf

region_gdf = filter_subregions(subregions, REGION)

geojson_data = json.loads(
    region_gdf.to_json()
)

for feature in geojson_data["features"]:

    props = feature["properties"]

    feature["properties"]["tooltip"] = f"""
    <div style="font-size:13px;">
        <b>Country:</b>
        {props.get("country", "Unknown")}
        <b>Region:</b>
        {props.get("bcr", "Unknown")}
    </div>
    """

region_layer = GeoJSON(

    data=geojson_data,

    style={
        "color": "white",
        "weight": 1.5,
        "fillColor": "white",
        "fillOpacity": 0.05,
        "opacity": 0.7,
    },

    hover_style={
        "color": "#00FFFF",
        "weight": 3,
        "fillColor": "#00FFFF",
        "fillOpacity": 0.20,
    },

    name="Region Boundaries"
)

def on_hover(event=None, feature=None, properties=None, **kwargs):
    tooltip_html = properties.get(
        "tooltip",
        ""
    )
    region_layer.popup = HTML(
        value=tooltip_html
    )

region_layer.on_hover(on_hover)

m.add(mean_density)
m.add(region_layer)
m.add(FullScreenControl())
m.add(LayersControl(collapsed=False, position="topright"))
m.add(ScaleControl(position="bottomleft"))

display(m)

Map(center=[55, -106], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…